# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
from google.colab import userdata
from huggingface_hub import login, list_repo_files, hf_hub_download
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
import os

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

def load_month_agg(month_str):
    files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
    month_files = [f for f in files if f"fact_content_daily_performance/month={month_str}" in f]
    local_paths = [
        hf_hub_download(repo_id="FlyRank/internship-warehouse", filename=f, repo_type="dataset")
        for f in month_files
    ]
    raw = pd.concat([pd.read_parquet(p) for p in local_paths], ignore_index=True)
    gsc = raw[raw['gsc_data_available'] == True].copy()
    agg = gsc.groupby(['client_hash_id', 'content_hash_id']).agg(
        days_with_data=('report_date', 'nunique'),
        total_impressions=('gsc_impressions', 'sum'),
        total_clicks=('gsc_clicks', 'sum'),
        avg_position=('gsc_avg_position', 'mean')
    ).reset_index()
    agg = agg[agg['total_impressions'] > 0].copy()
    agg['ctr'] = agg['total_clicks'] / agg['total_impressions']
    print(f"{month_str}: raw={raw.shape}, gsc-available={len(gsc)}, content-level={agg.shape}")
    return agg

march = load_month_agg("2026-03")
april = load_month_agg("2026-04")

2026-03: raw=(9841378, 30), gsc-available=3611061, content-level=(176738, 7)
2026-04: raw=(10424730, 30), gsc-available=3901060, content-level=(194760, 7)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [6]:
pos_bins = [0, 3, 10, 20, 50, 1000]
pos_labels = ['1-3', '4-10', '11-20', '21-50', '51+']

def add_ctr_gap(agg):
    agg = agg.copy()
    agg['position_bucket'] = pd.cut(agg['avg_position'], bins=pos_bins, labels=pos_labels)
    bucket_totals = agg.groupby('position_bucket', observed=True).agg(
        clicks_sum=('total_clicks', 'sum'), impressions_sum=('total_impressions', 'sum')
    )
    bucket_totals['expected_ctr'] = bucket_totals['clicks_sum'] / bucket_totals['impressions_sum']
    agg['expected_ctr'] = agg['position_bucket'].map(bucket_totals['expected_ctr']).astype(float)
    agg['ctr'] = agg['ctr'].astype(float)
    agg['ctr_gap'] = agg['expected_ctr'] - agg['ctr']
    agg['ctr_ratio'] = agg['ctr'] / agg['expected_ctr'].replace(0, np.nan)
    return agg

march = add_ctr_gap(march)
april = add_ctr_gap(april)

# April label: still underperforming next month
april['label_still_underperforming'] = (
    (april['total_impressions'] >= 1000) & (april['ctr_ratio'] < 0.5)
).astype(int)
print("April base rate (label mean):", april['label_still_underperforming'].mean().round(4))

# --- ADDITION: reproduce the ML-07 baseline rule on March, needed for Section 3 comparison ---
def score_row(row):
    if row['total_impressions'] >= 1000 and pd.notnull(row['ctr_ratio']) and row['ctr_ratio'] < 0.5:
        return row['ctr_gap'] * row['total_impressions']
    return 0

march['baseline_score'] = march.apply(score_row, axis=1)
print("March rows with baseline_score > 0:", (march['baseline_score'] > 0).sum())

April base rate (label mean): 0.0921
March rows with baseline_score > 0: 17902


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:
merged = march.merge(
    april[['client_hash_id', 'content_hash_id', 'label_still_underperforming']],
    on=['client_hash_id', 'content_hash_id'],
    how='inner'   # only content present in both months
)
print(f"March content items: {len(march)}, matched to April: {len(merged)}")
print("Dropped (no April match):", len(march) - len(merged))
print("baseline_score present:", 'baseline_score' in merged.columns)

feature_cols = ['total_impressions', 'total_clicks', 'avg_position', 'ctr',
                 'ctr_gap', 'ctr_ratio', 'days_with_data']
merged_model = merged.dropna(subset=feature_cols + ['label_still_underperforming']).copy()
print("Rows usable for modeling (no NaN features):", len(merged_model))

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(merged_model, groups=merged_model['client_hash_id']))
train_df = merged_model.iloc[train_idx]
test_df = merged_model.iloc[test_idx]

print(f"Train: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df)} rows, {test_df['client_hash_id'].nunique()} clients")
print("Overlap in clients between train/test:",
      len(set(train_df['client_hash_id']) & set(test_df['client_hash_id'])))

March content items: 176738, matched to April: 158549
Dropped (no April match): 18189
baseline_score present: True
Rows usable for modeling (no NaN features): 157790
Train: 136739 rows, 36 clients
Test:  21051 rows, 10 clients
Overlap in clients between train/test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [8]:
X_train, y_train = train_df[feature_cols], train_df['label_still_underperforming']
X_test, y_test = test_df[feature_cols], test_df['label_still_underperforming']

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

test_df = test_df.copy()
test_df['model_score'] = model.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()
print(f"Base rate (label mean) on test set: {base_rate:.3f}")
print(f"Test AUC: {roc_auc_score(y_test, test_df['model_score']):.3f}")
print()

results = []
for k in [20, 50, 100]:
    p_baseline = precision_at_k(test_df['baseline_score'], test_df['label_still_underperforming'], k)
    p_model = precision_at_k(test_df['model_score'], test_df['label_still_underperforming'], k)
    results.append({'k': k, 'baseline_precision': p_baseline, 'model_precision': p_model, 'base_rate': base_rate})

comparison_table = pd.DataFrame(results)
print(comparison_table)

Base rate (label mean) on test set: 0.102
Test AUC: 0.832

     k  baseline_precision  model_precision  base_rate
0   20                0.95             0.95    0.10218
1   50                0.98             0.80    0.10218
2  100                0.92             0.78    0.10218


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
coefs = pd.Series(model.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
print("Feature coefficients (standardized effect on log-odds):")
print(coefs)

test_df['error'] = test_df['model_score'] - test_df['label_still_underperforming']
worst_false_positives = test_df.sort_values('model_score', ascending=False).query('label_still_underperforming == 0').head(3)
worst_false_negatives = test_df.sort_values('model_score', ascending=True).query('label_still_underperforming == 1').head(3)

print("\nTop false positives (model confident, but page recovered by April):")
print(worst_false_positives[['content_hash_id', 'model_score', 'total_impressions', 'ctr_ratio']].to_string(index=False))

print("\nTop false negatives (model missed, but page still underperforming in April):")
print(worst_false_negatives[['content_hash_id', 'model_score', 'total_impressions', 'ctr_ratio']].to_string(index=False))

Feature coefficients (standardized effect on log-odds):
total_clicks        -0.175846
days_with_data       0.070738
ctr_ratio           -0.036522
avg_position        -0.030387
ctr_gap             -0.006741
ctr                 -0.001135
total_impressions    0.000645
dtype: float64

Top false positives (model confident, but page recovered by April):
         content_hash_id  model_score  total_impressions  ctr_ratio
content_d1db17521a55d9fc     1.000000              44650   0.623076
content_e3496dac741da4f9     0.999999              63494   0.753693
content_567d370cf1fdbd1d     0.999994              25527   0.573600

Top false negatives (model missed, but page still underperforming in April):
         content_hash_id  model_score  total_impressions  ctr_ratio
content_4de3cc4157dfe37b     0.008470              25656   1.444068
content_068f29cb110b6cc7     0.010529               2364   3.019631
content_36985c86fb5f660c     0.017939               2092   2.941587


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.